In [3]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision, layers, models, optimizers, losses, metrics
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryFocalCrossentropy
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, auc, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import glob
import time
import csv
from datetime import datetime
import matplotlib.pyplot as plt
import random
import pandas as pd

# Reduce TF log messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Enable GPU memory growth (important on laptops)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

# Use mixed precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision enabled:", mixed_precision.global_policy())

2026-03-22 13:29:33.111222: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-22 13:29:33.289819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-22 13:29:33.363287: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-22 13:29:33.389833: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-22 13:29:33.520274: I tensorflow/core/platform/cpu_feature_guar

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, compute capability 12.0
Mixed precision enabled: <Policy "mixed_float16">


I0000 00:00:1774186178.489570     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186179.132454     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186179.139119     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186179.144617     164 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [4]:
CLASS_NAMES = ["Normal", "Abnormal"]

SEED = 0
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

CSV_FILE = "MyBinaryCNN_experiments_log.csv"
EXPERIMENTS_FOLDER = "experiments_MyBinaryCNN"
CURVE_PLOT_FOLDER = os.path.join(EXPERIMENTS_FOLDER, "loss_plots")
ROC_PLOT_FOLDER = os.path.join(EXPERIMENTS_FOLDER, "roc_plots")

# Ensure folders exist
os.makedirs(CURVE_PLOT_FOLDER, exist_ok=True)
os.makedirs(ROC_PLOT_FOLDER, exist_ok=True)
os.makedirs(EXPERIMENTS_FOLDER, exist_ok=True)

In [5]:
# Configurable parameters
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Training parameters
EPOCHS = 30
EARLY_STOP_PATIENCE = 5
BASE_LR = 1e-5

# Callbacks
lr_reduce = ReduceLROnPlateau(
    monitor="val_roc_auc",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_roc_auc",
    patience=EARLY_STOP_PATIENCE,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

# Data paths
TRAIN_IMAGES_PATH = "images_bal_bin.npy"
TRAIN_LABELS_PATH = "labels_bal_bin.npy"

VAL_IMAGES_PATH = "validation_data_norm.npy"
VAL_LABELS_PATH = "validation_labels_norm.npy"

TEST_IMAGES_PATH = "test_data_norm.npy"
TEST_LABELS_PATH = "test_labels_norm.npy"

In [6]:
# Load numpy arrays
X_train = np.load(TRAIN_IMAGES_PATH)
y_train = np.load(TRAIN_LABELS_PATH)

# X_val = np.load(VAL_IMAGES_PATH) 
# y_val = np.load(VAL_LABELS_PATH)

# To facilitate the final test, change the data used only instead of all the variables
X_val = np.load(TEST_IMAGES_PATH) 
y_val = np.load(TEST_LABELS_PATH)

# Relabel: 0:0, 1:1, 2:1
y_train = (y_train > 0).astype(np.float32)
y_val   = (y_val > 0).astype(np.float32)

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Train class balance:", np.mean(y_train))
print("Val class balance:", np.mean(y_val))

Train shape: (14578, 299, 299, 1) (14578,)
Val shape: (7682, 299, 299, 1) (7682,)
Train class balance: 0.5
Val class balance: 0.12822181


In [7]:
# Preprocessing function
def preprocess_image(img, label, augment=False):
    img = tf.cast(img, tf.float32)
    
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.05)
        img = tf.image.random_contrast(img, 0.95, 1.05)

    return img, label

# tf.data pipelines
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(2048, seed=SEED)
    .map(lambda x, y: preprocess_image(x, y, augment=True), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = (
    val_ds
    .map(lambda x, y: preprocess_image(x, y, augment=False), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


I0000 00:00:1774186310.535270     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186310.542040     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186310.544566     164 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1774186310.698555     164 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [8]:
# Check one batch
for images, labels in train_ds.take(1):
    print("Train batch images shape:", images.shape)
    print("Train batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())

for images, labels in val_ds.take(1):
    print("Validation batch images shape:", images.shape)
    print("Validation batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())


Train batch images shape: (32, 299, 299, 1)
Train batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'float32'>
Image min/max: -0.040607035 0.95444405


2026-03-22 13:32:08.516950: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Validation batch images shape: (32, 299, 299, 1)
Validation batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'float32'>
Image min/max: 0.0 0.8862745


2026-03-22 13:32:09.718285: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
# Model architecture
model = models.Sequential([
    layers.Input(shape=(299, 299, 1)),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    # layers.Conv2D(512, 3, padding="same", activation="relu"),
    # layers.Conv2D(512, 3, padding="same", activation="relu"),
    # layers.MaxPooling2D(),
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")   
])

# # Original Model
# model = models.Sequential([
#     layers.Input(shape=(299, 299, 1)),

#     layers.Conv2D(32, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(64, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(128, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(256, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.GlobalAveragePooling2D(),
#     layers.Dropout(0.4),

#     layers.Dense(1, activation="sigmoid")
# ])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
    loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=3.0, alpha=0.5),
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc")]
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 299, 299, 32)      320       
                                                                 
 conv2d_1 (Conv2D)           (None, 299, 299, 32)      9248      
                                                                 
 max_pooling2d (MaxPooling2  (None, 149, 149, 32)      0         
 D)                                                              
                                                                 
 conv2d_2 (Conv2D)           (None, 149, 149, 64)      18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 149, 149, 64)      36928     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 74, 74, 64)        0         
 g2D)                                                   

In [10]:
# Callbacks
callbacks = [early_stop, lr_reduce]

# Time model training
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_time_sec = time.time() - start_time
print(f"Training time (min): {training_time_sec / 60:.2f}")

Epoch 1/30


2026-03-22 13:32:19.564909: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
2026-03-22 13:32:19.778591: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90700
W0000 00:00:1774186339.955477     361 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186339.957437     361 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186339.970935     361 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186339.973125     361 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced

  1/456 [..............................] - ETA: 1:24:19 - loss: 0.0867 - roc_auc: 0.5000

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1774186347.177008     347 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186347.177715     347 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186347.179201     347 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be 

  2/456 [..............................] - ETA: 19:49 - loss: 0.0867 - roc_auc: 0.5000  

...

456/456 [==============================] - 31s 69ms/step - loss: 0.0700 - roc_auc: 0.7976 - val_loss: 0.0739 - val_roc_auc: 0.7995 - lr: 1.0000e-05
Epoch 4/30
455/456 [============================>.] - ETA: 0s - loss: 0.0698 - roc_auc: 0.7987  

W0000 00:00:1774186474.652473     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.653216     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.653899     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.654581     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.655218     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.655901     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.656615     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.657345     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1774186474.658022     349 gp

456/456 [==============================] - ETA: 0s - loss: 0.0698 - roc_auc: 0.7987

W0000 00:00:1774186474.874092     349 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced


456/456 [==============================] - 32s 69ms/step - loss: 0.0698 - roc_auc: 0.7987 - val_loss: 0.0707 - val_roc_auc: 0.8024 - lr: 1.0000e-05
Epoch 5/30
456/456 [==============================] - 31s 69ms/step - loss: 0.0694 - roc_auc: 0.8015 - val_loss: 0.0739 - val_roc_auc: 0.8038 - lr: 1.0000e-05
Epoch 6/30
456/456 [==============================] - 31s 68ms/step - loss: 0.0691 - roc_auc: 0.8029 - val_loss: 0.0789 - val_roc_auc: 0.8071 - lr: 1.0000e-05
Epoch 7/30
456/456 [==============================] - 31s 69ms/step - loss: 0.0685 - roc_auc: 0.8063 - val_loss: 0.0661 - val_roc_auc: 0.8095 - lr: 1.0000e-05
Epoch 8/30
456/456 [==============================] - 31s 69ms/step - loss: 0.0680 - roc_auc: 0.8103 - val_loss: 0.0610 - val_roc_auc: 0.8159 - lr: 1.0000e-05
Epoch 9/30
456/456 [==============================] - 31s 69ms/step - loss: 0.0677 - roc_auc: 0.8120 - val_loss: 0.0701 - val_roc_auc: 0.8133 - lr: 1.0000e-05
Epoch 10/30
456/456 [==============================] - 31

In [13]:
# Validation predictions
val_ds_eval = val_ds.unbatch().batch(32)
y_val_bin = np.concatenate([y for x, y in val_ds], axis=0).ravel()
y_pred_probs = model.predict(val_ds_eval).ravel()

# Default threshold = 0.5
y_pred = (y_pred_probs >= 0.485).astype(int) # changed from default to 0.485

# ROC-AUC
roc_auc = roc_auc_score(y_val_bin, y_pred_probs)
print(f"\nROC-AUC: {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_val_bin, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
print("\nConfusion matrix:\n", cm)
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Classification report
report_dict = classification_report(
    y_val_bin,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)
print("\nClassification report:\n", classification_report(y_val_bin, y_pred, target_names=CLASS_NAMES))

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")

241/241 [==============================] - 6s 18ms/step

ROC-AUC: 0.9050

Confusion matrix:
 [[4974 1723]
 [  85  900]]

Normalized confusion matrix:
 [[0.74 0.26]
 [0.09 0.91]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.98      0.74      0.85      6697
    Abnormal       0.34      0.91      0.50       985

    accuracy                           0.76      7682
   macro avg       0.66      0.83      0.67      7682
weighted avg       0.90      0.76      0.80      7682

Accuracy for class 0 (Normal): 0.7427
Accuracy for class 1 (Abnormal): 0.9137


2026-03-22 16:21:21.667601: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 13312982648243064943


In [12]:
# Threshold analysis no longer required once threshold has been chosen - 0.485
# It is performed anyway so as to not interfere with subsequent experiment logging

# Threshold analysis
thresholds = np.linspace(0.0, 1.0, 201)  # step=0.005
rows = []

for t in thresholds:
    y_pred_t = (y_pred_probs >= t).astype(int)
    cm_t = confusion_matrix(y_val_bin, y_pred_t)
    
    # Skip degenerate thresholds
    if cm_t.shape != (2, 2):
        continue

    tn, fp, fn, tp = cm_t.ravel()
    
    # Per-class accuracy
    acc_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
    acc_abnormal = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Overall metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision_abn = precision_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)
    recall_abn = recall_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)
    f1_abn = f1_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)

    rows.append({
        "threshold": t,
        "accuracy_overall": accuracy,
        "acc_normal": acc_normal,
        "acc_abnormal": acc_abnormal,
        "precision_abnormal": precision_abn,
        "recall_abnormal": recall_abn,
        "f1_abnormal": f1_abn,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    })

df_thresholds = pd.DataFrame(rows)

# Apply example constraints
filtered = df_thresholds[
    (df_thresholds["acc_normal"] >= 0.75) &
    (df_thresholds["acc_abnormal"] >= 0.9)
].copy()

print(f"\nThresholds satisfying constraints: {len(filtered)}")

# Sort by metrics of interest
filtered = filtered.sort_values(by=["recall_abnormal", "f1_abnormal"], ascending=False)

pd.set_option("display.float_format", "{:.3f}".format)
print("\nTop thresholds:\n", filtered.head(10))


Thresholds satisfying constraints: 1

Top thresholds:
     threshold  accuracy_overall  acc_normal  acc_abnormal  precision_abnormal  \
92      0.460             0.772       0.753         0.902               0.349   

    recall_abnormal  f1_abnormal    tn    fp  fn   tp  
92            0.902        0.503  5041  1656  97  888  


In [13]:
# Auto-increment experiment ID
def get_next_experiment_id(csv_file):
    if not os.path.exists(csv_file):
        return 1
    with open(csv_file, "r") as f:
        return sum(1 for _ in f)

# Extract max L1/L2 across layers
def extract_l1_l2(model):
    l1_vals = []
    l2_vals = []
    for layer in model.layers:
        reg = getattr(layer, "kernel_regularizer", None)
        if reg:
            if hasattr(reg, "l1") and reg.l1 is not None: 
                l1_vals.append(float(tf.keras.backend.get_value(reg.l1)))
            if hasattr(reg, "l2") and reg.l2 is not None: 
                l2_vals.append(float(tf.keras.backend.get_value(reg.l2)))
    return (
        max(l1_vals) if l1_vals else 0.0, 
        max(l2_vals) if l2_vals else 0.0)

# Extract max dropout rate
def extract_max_dropout(model):
    rates = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dropout):
            rates.append(float(layer.rate))
    return max(rates) if rates else 0.0

# Build experiment row
exp_id = get_next_experiment_id(CSV_FILE)
l1_val, l2_val = extract_l1_l2(model)
dropout_val = extract_max_dropout(model)
learning_rate_val = float(model.optimizer.learning_rate.numpy())
train_loss = history.history["loss"][-1]
val_loss = history.history["val_loss"][-1]

row = {
    "experiment_id": exp_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "task": "binary roc-auc",
    # "backbone": BACKBONE_NAME,
    # "pretrained": PRETRAINED,
    # "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    # "shuffle_buffer": SHUFFLE_BUFFER,
    "epochs": len(history.history["loss"]),
    "learning_rate": learning_rate_val,
    "dropout_rate": dropout_val,
    "l1": l1_val,
    "l2": l2_val,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "training_time_min": training_time_sec / 60.0,
    "roc_auc": roc_auc,  # from report cell
    "num_thresholds_meeting_constraints": len(filtered)
}

# Overall accuracy
overall_acc = np.mean((y_pred_probs >= 0.5).astype(int) == y_val_bin)
row["accuracy"] = overall_acc

# Per-class metrics
for i, cls in enumerate(CLASS_NAMES):
    cls_l = cls.lower()
    row[f"acc_{cls_l}"] = per_class_acc[i]
    row[f"precision_{cls_l}"] = report_dict[cls]["precision"]
    row[f"recall_{cls_l}"] = report_dict[cls]["recall"]
    row[f"f1_{cls_l}"] = report_dict[cls]["f1-score"]

# Save to CSV
file_exists = os.path.isfile(CSV_FILE)
with open(CSV_FILE, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Experiment {exp_id} logged to {CSV_FILE}")

# Save training curves
plt.figure(figsize=(8,6))
plt.plot(history.history["loss"], label="Train Loss", marker='o')
plt.plot(history.history["val_loss"], label="Val Loss", marker='o')
if "accuracy" in history.history:
    plt.plot(history.history["accuracy"], label="Train Acc", marker="x")
    plt.plot(history.history["val_accuracy"], label="Val Acc", marker="x")
plt.title(f"Training Curves - Experiment {exp_id:05d}")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(CURVE_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()

# Save ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_val_bin, y_pred_probs)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc_val:.3f})")
plt.plot([0,1], [0,1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - Experiment {exp_id:05d}")
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(os.path.join(ROC_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()
print(f"ROC curve saved to {ROC_PLOT_FOLDER}")


Experiment 2 logged to MyBinaryCNN_experiments_log.csv
ROC curve saved to experiments_MyBinaryCNN/roc_plots
